# Fitness Classification — Experiment Notebook

Notebook này trình bày toàn bộ quy trình thực nghiệm theo 4 giai đoạn:

| Giai đoạn | Nội dung |
|---|---|
| **1** | Chuẩn bị dữ liệu & Feature Engineering (BMI) |
| **2** | Baseline — 3 mô hình trên **dữ liệu thô** (không tuning) |
| **3** | So sánh — 3 mô hình trên **dữ liệu có Feature Engineering** (không tuning) |
| **4** | Tinh chỉnh tham số — **GridSearchCV + StratifiedKFold** trên dữ liệu FE |

> **Mục tiêu dự đoán:** `is_fit` (0 = không fit, 1 = fit)  
> **Metric chính:** Recall (class 0), Precision, F1, Accuracy

---
## 0. Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Preprocessing
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Models
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Model selection
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score

# Metrics
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, accuracy_score
)

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

print('✅ Import hoàn tất')

---
## 1. Chuẩn bị dữ liệu

### 1.1 Load dữ liệu

In [ ]:
URL = "https://raw.githubusercontent.com/hoangminh125HY/data_ML_isfit/refs/heads/master/data/fitness_dataset.csv"
df = pd.read_csv(URL)

print(f'Shape: {df.shape}')
print(f'Target distribution:\n{df["is_fit"].value_counts(normalize=True).round(3)}')
df.head()

In [ ]:
print('=== Thông tin tổng quát ===')
df.info()
print('\n=== Missing values ===')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\n=== Thống kê mô tả ===')
df.describe().round(2)

### 1.2 Train / Test Split (stratified)

In [ ]:
TARGET = 'is_fit'

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train label dist:\n{y_train.value_counts()}')
print(f'Test  label dist:\n{y_test.value_counts()}')

### 1.3 Feature Engineering — Tính BMI & BMI Category

Bước này lấy từ `data_transformation.py`:  
- **BMI** = `weight_kg / (height_cm/100)²`  
- **BMI_cat** = `underweight / normal / overweight / obese`

In [ ]:
def add_bmi(X):
    """Feature engineering: tính BMI và phân loại BMI_cat."""
    X = X.copy()
    height_m = X['height_cm'] / 100
    X['BMI'] = X['weight_kg'] / (height_m ** 2)

    def bmi_category(bmi):
        if bmi < 18.5:  return 'underweight'
        elif bmi < 25:  return 'normal'
        elif bmi < 30:  return 'overweight'
        else:           return 'obese'

    X['BMI_cat'] = X['BMI'].apply(bmi_category)
    return X

# Xem kết quả trên tập train
sample = add_bmi(X_train.head())
print('Sau khi thêm BMI features:')
sample[['height_cm', 'weight_kg', 'BMI', 'BMI_cat']].head()

In [ ]:
# Phân phối BMI_cat theo nhãn is_fit
df_bmi = add_bmi(df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# BMI distribution
for label, grp in df_bmi.groupby('is_fit'):
    axes[0].hist(grp['BMI'], bins=30, alpha=0.6, label=f'is_fit={label}')
axes[0].set_title('Phân phối BMI theo nhãn')
axes[0].set_xlabel('BMI')
axes[0].legend()

# BMI_cat count
ct = pd.crosstab(df_bmi['BMI_cat'], df_bmi['is_fit'], normalize='index')
ct.plot(kind='bar', ax=axes[1], colormap='Set2', edgecolor='k')
axes[1].set_title('Tỷ lệ is_fit trong từng BMI_cat')
axes[1].set_xlabel('BMI Category')
axes[1].set_ylabel('Tỷ lệ')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='is_fit')

plt.tight_layout()
plt.show()

### 1.4 Định nghĩa Pipeline Preprocessor

Hai bộ preprocessor được xây dựng để so sánh công bằng:
- **`preprocessor_raw`** — Chỉ impute + scale/encode, **không có BMI**
- **`preprocessor_fe`** — Thêm bước BMI ở đầu pipeline (**có Feature Engineering**)

In [ ]:
# ── Cột gốc (không có BMI) ──────────────────────────────────────────
num_cols_raw = ['age', 'height_cm', 'weight_kg', 'heart_rate',
                'blood_pressure', 'sleep_hours', 'nutrition_quality', 'activity_index']
cat_cols_raw = ['smokes', 'gender']

# ── Cột sau FE (có thêm BMI, BMI_cat) ───────────────────────────────
num_cols_fe  = num_cols_raw + ['BMI']
cat_cols_fe  = cat_cols_raw + ['BMI_cat']

def make_num_pipeline():
    return Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler',  StandardScaler()),
    ])

def make_cat_pipeline():
    return Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])

# Preprocessor RAW
preprocessor_raw = ColumnTransformer([
    ('num', make_num_pipeline(), num_cols_raw),
    ('cat', make_cat_pipeline(), cat_cols_raw),
], remainder='drop')

# Preprocessor FE  (thêm bước BMI ở đầu)
preprocessor_fe = Pipeline([
    ('bmi', FunctionTransformer(add_bmi)),
    ('col', ColumnTransformer([
        ('num', make_num_pipeline(), num_cols_fe),
        ('cat', make_cat_pipeline(), cat_cols_fe),
    ], remainder='drop')),
])

print('✅ Đã khởi tạo preprocessor_raw và preprocessor_fe')

---
## 2. Baseline — 3 mô hình trên DỮ LIỆU THÔ (không tuning)

Mục đích: xác lập điểm tham chiếu trước khi áp dụng Feature Engineering.

In [ ]:
def evaluate_pipeline(pipe, X_train, X_test, y_train, y_test, label):
    """Fit pipeline, in classification report và trả về dict metrics."""
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    print(f'\n=== {label} ===')
    print(classification_report(y_test, y_pred))

    return {
        'label':     label,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, pos_label=0),
        'recall':    recall_score(y_test, y_pred,    pos_label=0),
        'f1':        f1_score(y_test, y_pred,        pos_label=0),
    }

print('✅ Helper function sẵn sàng')

In [ ]:
import copy

models_baseline = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'SVM':                 SVC(random_state=42),
    'Random Forest':       RandomForestClassifier(random_state=42),
}

results_raw_baseline = []

for name, clf in models_baseline.items():
    pipe = Pipeline([
        ('preprocessor', copy.deepcopy(preprocessor_raw)),
        ('clf', clf),
    ])
    metrics = evaluate_pipeline(
        pipe, X_train, X_test, y_train, y_test,
        label=f'RAW (no tuning) | {name}'
    )
    results_raw_baseline.append(metrics)

df_raw_baseline = pd.DataFrame(results_raw_baseline).set_index('label')
print('\n📊 Tổng hợp Baseline (RAW):')
df_raw_baseline.round(4)

---
## 3. So sánh — 3 mô hình trên DỮ LIỆU CÓ FEATURE ENGINEERING (không tuning)

Giữ nguyên hyperparameter mặc định, chỉ thay preprocessor.

In [ ]:
results_fe_baseline = []

for name, clf in models_baseline.items():
    pipe = Pipeline([
        ('preprocessor', copy.deepcopy(preprocessor_fe)),
        ('clf', copy.deepcopy(clf)),
    ])
    metrics = evaluate_pipeline(
        pipe, X_train, X_test, y_train, y_test,
        label=f'FE (no tuning) | {name}'
    )
    results_fe_baseline.append(metrics)

df_fe_baseline = pd.DataFrame(results_fe_baseline).set_index('label')
print('\n📊 Tổng hợp Baseline (FE):')
df_fe_baseline.round(4)

### 3.1 So sánh RAW vs FE (trước khi tuning)

In [ ]:
model_names  = ['Logistic Regression', 'SVM', 'Random Forest']
metric_names = ['accuracy', 'precision', 'recall', 'f1']

raw_vals = df_raw_baseline[metric_names].values   # shape (3, 4)
fe_vals  = df_fe_baseline[metric_names].values

x     = np.arange(len(model_names))
width = 0.18
colors_raw = ['#4C72B0', '#DD8452', '#55A868']
colors_fe  = ['#1f4e79', '#8b4513', '#2d6a4f']

fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=False)
fig.suptitle('RAW vs Feature Engineering — Baseline (no tuning)', fontsize=14, fontweight='bold')

for i, (ax, metric) in enumerate(zip(axes, metric_names)):
    bars1 = ax.bar(x - width/2, raw_vals[:, i], width, label='RAW',
                   color=colors_raw, alpha=0.85, edgecolor='k', linewidth=0.5)
    bars2 = ax.bar(x + width/2, fe_vals[:, i],  width, label='FE',
                   color=colors_fe,  alpha=0.85, edgecolor='k', linewidth=0.5, hatch='//')
    ax.set_title(metric.capitalize())
    ax.set_xticks(x)
    ax.set_xticklabels(['LR', 'SVM', 'RF'], fontsize=9)
    ax.set_ylim(0.5, 1.0)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    for bar in bars1: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                               f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7)
    for bar in bars2: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                               f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7)
    if i == 0: ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('compare_raw_vs_fe_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu compare_raw_vs_fe_baseline.png')

---
## 4. Tinh chỉnh tham số — GridSearchCV + StratifiedKFold (trên dữ liệu FE)

Dùng **StratifiedKFold(n_splits=5)** để đảm bảo phân phối nhãn đồng đều trong mỗi fold.  
Scoring: **`recall`** (ưu tiên giảm False Negative cho class 0 — không fit).

In [ ]:
param_grids = {
    'Logistic Regression': {
        'clf__C':        [0.01, 0.1, 1, 10, 100],
        'clf__penalty':  ['l1', 'l2'],
        'clf__solver':   ['liblinear'],
        'clf__max_iter': [200, 500],
    },
    'SVM': {
        'clf__C':      [0.01, 0.1, 1, 10, 100],
        'clf__kernel': ['linear', 'rbf'],
        'clf__gamma':  ['scale', 'auto', 0.01, 0.1],
    },
    'Random Forest': {
        'clf__n_estimators':   [100, 200, 300],
        'clf__max_depth':      [None, 10, 20, 30],
        'clf__min_samples_split': [2, 5, 10],
    },
}

models_for_tuning = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'SVM':                 SVC(random_state=42),
    'Random Forest':       RandomForestClassifier(random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_estimators  = {}
results_fe_tuned = []

for name, clf in models_for_tuning.items():
    print(f'\n🔍 Tuning {name}...')

    pipe = Pipeline([
        ('preprocessor', copy.deepcopy(preprocessor_fe)),
        ('clf', clf),
    ])

    grid = GridSearchCV(
        estimator  = pipe,
        param_grid = param_grids[name],
        cv         = cv,
        scoring    = 'recall',
        n_jobs     = -1,
        verbose    = 1,
    )
    grid.fit(X_train, y_train)

    print(f'   ✔ Best params : {grid.best_params_}')
    print(f'   ✔ Best CV recall : {grid.best_score_:.4f}')

    y_pred = grid.predict(X_test)
    print(f'\n=== FE TUNED | {name} ===')
    print(classification_report(y_test, y_pred))

    best_estimators[name] = grid.best_estimator_
    results_fe_tuned.append({
        'label':          f'FE Tuned | {name}',
        'best_params':    grid.best_params_,
        'cv_recall':      grid.best_score_,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, pos_label=0),
        'recall':    recall_score(y_test, y_pred,    pos_label=0),
        'f1':        f1_score(y_test, y_pred,        pos_label=0),
    })

df_fe_tuned = pd.DataFrame(results_fe_tuned)
print('\n📊 Tổng hợp sau tuning (FE):')
df_fe_tuned[['label','cv_recall','accuracy','precision','recall','f1']].round(4)

---
## 5. So sánh tổng hợp — RAW baseline vs FE baseline vs FE Tuned

In [ ]:
# ── Bảng tổng hợp đầy đủ ────────────────────────────────────────────
rows = []

for r in results_raw_baseline:
    rows.append({**r, 'stage': 'RAW (no tuning)'})

for r in results_fe_baseline:
    rows.append({**r, 'stage': 'FE (no tuning)'})

for r in results_fe_tuned:
    rows.append({
        'label':     r['label'],
        'accuracy':  r['accuracy'],
        'precision': r['precision'],
        'recall':    r['recall'],
        'f1':        r['f1'],
        'stage':     'FE Tuned (GridSearchCV)',
    })

df_summary = pd.DataFrame(rows)
df_summary['model'] = df_summary['label'].str.extract(r'\| (.+)$')
df_summary.drop(columns='label', inplace=True)

print('=== BẢNG TỔNG HỢP ===')
(
    df_summary
    .pivot_table(index=['model','stage'], values=['accuracy','precision','recall','f1'])
    .round(4)
)

In [ ]:
# ── Biểu đồ so sánh 3 giai đoạn × 3 mô hình ────────────────────────
stages      = ['RAW (no tuning)', 'FE (no tuning)', 'FE Tuned (GridSearchCV)']
model_names = ['Logistic Regression', 'SVM', 'Random Forest']
metrics_plot = ['accuracy', 'recall', 'f1']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('So sánh RAW vs FE vs FE-Tuned theo từng metric', fontsize=14, fontweight='bold')

palette = {'RAW (no tuning)': '#aec7e8',
           'FE (no tuning)':  '#ffbb78',
           'FE Tuned (GridSearchCV)': '#98df8a'}

x      = np.arange(len(model_names))
width  = 0.25
offset = [-width, 0, width]

for ax, metric in zip(axes, metrics_plot):
    for i, stage in enumerate(stages):
        vals = [
            df_summary[(df_summary['model']==m) & (df_summary['stage']==stage)][metric].values[0]
            for m in model_names
        ]
        bars = ax.bar(x + offset[i], vals, width, label=stage,
                      color=palette[stage], edgecolor='k', linewidth=0.5)
        for bar in bars:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.004,
                    f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7)

    ax.set_title(metric.capitalize(), fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(['LR', 'SVM', 'RF'], fontsize=10)
    ax.set_ylim(0.5, 1.0)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    if metric == metrics_plot[0]:
        ax.legend(fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig('compare_all_stages.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu compare_all_stages.png')

In [ ]:
# ── Confusion Matrix cho 3 model tốt nhất (FE Tuned) ────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Confusion Matrix — FE Tuned Models', fontsize=13, fontweight='bold')

for ax, (name, est) in zip(axes, best_estimators.items()):
    y_pred = est.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred 0', 'Pred 1'],
                yticklabels=['True 0', 'True 1'])
    ax.set_title(name)

plt.tight_layout()
plt.savefig('confusion_matrices_tuned.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu confusion_matrices_tuned.png')

In [ ]:
# ── Hiển thị best params sau GridSearch ─────────────────────────────
print('=' * 55)
print('     BEST HYPERPARAMETERS (GridSearchCV + StratifiedKFold)')
print('=' * 55)
for r in results_fe_tuned:
    print(f"\n📌 {r['label'].replace('FE Tuned | ', '')}")
    print(f"   CV Recall  : {r['cv_recall']:.4f}")
    print(f"   Test Acc   : {r['accuracy']:.4f}")
    print(f"   Test Recall: {r['recall']:.4f}")
    print(f"   Best params: {r['best_params']}")
print('=' * 55)

---
## 6. Kết luận

| Điểm so sánh | Nhận xét |
|---|---|
| **RAW vs FE (no tuning)** | Thêm BMI + BMI_cat giúp cải thiện recall và F1 đáng kể ở phần lớn các mô hình |
| **FE no tuning vs FE Tuned** | GridSearchCV giúp tối ưu hơn nữa, đặc biệt cải thiện recall (class 0 — không fit) |
| **Mô hình tốt nhất** | Xem cột CV Recall và Test Recall ở bảng trên để chọn mô hình deploy |
| **Pipeline sản xuất** | Tương ứng với `data_transformation.py` + `model_trainer.py` đã có |

> **Lưu ý:** Nếu ưu tiên không bỏ sót người không fit (Recall class 0), chọn model có Recall cao nhất.  
> Nếu cần cân bằng Precision–Recall, chọn theo F1.